# Simulating some NR events

Pueh Leng Tan, 10 March 2026

In [1]:
import os
from time import time

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax.numpy as jnp
import multihist as mh
import json

import appletree as apt
from appletree.utils import get_file_path
from appletree.share import _cached_functions

import aptext

XLA_PYTHON_CLIENT_PREALLOCATE is set to false
XLA_PYTHON_CLIENT_ALLOCATOR is set to platform
Using aptext package from https://github.com/XENONnT/applefiles


In [2]:
# constrain the GPU memory usage
apt.set_gpu_memory_usage(0.5)

## Define component

### ComponentSim

In [3]:
# Initialize component
#nr = apt.NR(bins=[bins_cs1, bins_cs2], bins_type="irreg")
#nr = apt.NR()
#nr = apt.components.NRBand2f()

# https://github.com/XENONnT/applefiles/blob/cbb3139150526e647d1e2985f2079577f8218cd3/aptext/components/three_fold.py#L57-L88
nr = apt.components.AmBe()

# i should write my own stuff (or use minghao's) under components of aptext of apt
# this component tells me exactly which steps to do, just like registering cuts in cutax
# https://github.com/XENONnT/applefiles/blob/cbb3139150526e647d1e2985f2079577f8218cd3/aptext/components/two_fold.py#L121-L146

In [4]:
aa = nr.show_config()
# all the ingredients it needs
# if current = None, means it uses default

In [5]:
aa

,option,default,current,applies_to,help
0,s1_bias_3f,_s1_bias.json,None,[s1_area],3fold S1 reconstruction bias
1,s1_smear_3f,_s1_smearing.json,None,[s1_area],3fold S1 reconstruction smearing
2,s1_correction,_s1_correction.json,None,[s1_correction],S1 xyz correction on reconstructed positions
3,s1_lce,_s1_correction.json,None,[s1_lce],S1 light collection efficiency
4,g4,"[g4_ambe_sr0.npy, 1000000, 7]",None,"[energy, x, y, z, time, s2_tag, event_id, g4_e...",full-chain geant4 input file
5,literature_field,23.0,None,[ThomasImel],Drift field in each literature
6,s2_bias,_s2_bias.json,None,[s2_area_naive],S2 reconstruction bias
7,s2_smear,_s2_smearing.json,None,[s2_area_naive],S2 reconstruction smearing
8,s2_bias,_s2_bias.json,None,[alt_s2_area_naive],S2 reconstruction bias
9,s2_smear,_s2_smearing.json,None,[alt_s2_area_naive],S2 reconstruction smearing


In [6]:
f_instruct = 'instruct_ambe_realistic.json'
with open(f_instruct, 'rb') as fid:
    instruct = json.load(fid) # dictionary

In [7]:
my_config = {} # get this from miao when using his component in the future

for _opt in aa['option']: # for everything that you can set
    if _opt in instruct['configs']: # if it's inside the instruction file
        my_config[_opt] = instruct['configs'][_opt] # use that

In [8]:
my_config

{'s1_bias_3f': '3fold_s1_bias_bkg_sr1.json',
 's1_smear_3f': '3fold_s1_smearing_bkg_sr1.json',
 's1_correction': 's1_correction_sr1.json',
 's1_lce': 's1_correction_sr1.json',
 'g4': ['g4_ambe_sr1.npy', 100000, 7],
 's2_bias': 's2_bias_bkg_sr1.json',
 's2_smear': 's2_smearing_bkg_sr1.json',
 'posrec_reso': 'posrec_reso_sr0.json',
 's2_lce': 's2_correction_sr1.json',
 'elife': 'elife_bkg_sr1.json',
 'civ': 'civ_sr1.json',
 's2_correction': 's2_correction_sr1.json',
 's1_eff_n_hits_3f': ['3fold_recon_eff_n_hits_median_sr1.json',
  '3fold_recon_eff_n_hits_lower_sr1.json',
  '3fold_recon_eff_n_hits_upper_sr1.json'],
 's1_cut_acc': ['/home/puehlengt/applefiles/files/cut_acceptance/s1_cut_acc_median_ambe_flat.json',
  '/home/puehlengt/applefiles/files/cut_acceptance/s1_cut_acc_lower_ambe_flat.json',
  '/home/puehlengt/applefiles/files/cut_acceptance/s1_cut_acc_upper_ambe_flat.json'],
 's2_cut_acc': ['/home/puehlengt/applefiles/files/cut_acceptance/s2_cut_acc_median_ambe_fitted.json',
  '/hom

In [9]:
# Deduce the workflow(datastructure)
nr.deduce(data_names=["cs1", "cs2"], func_name="simulate")  # 'eff'(efficiency) is always simulated
# above line basically says that i want cs1, cs2, go deduce and gather ingredients required to compute those

nr.set_config(my_config) # this must be done before nr.compile
nr.rate_name = "nr_rate"  # also we have to specify a normalization factor of the component

# Compile NR script
# This is meta-programing because  appletree can generate codes dynamically
nr.compile()

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_s1_bias_bkg_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_s1_bias_bkg_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_s1_smearing_bkg_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_s1_smearing_bkg_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load s1_correction_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/s1_correction_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load s1_correction_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/s1_correction_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/hom

There are 31621 events, 98363 main/alt S2, which are not equal to batch_size.


/home/puehlengt/applefiles/aptext/multiscatter/g4.py:105: UserWarning: Efficiency is not provided in /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/g4_ambe_sr1.npy, set to 1.
  warn(f"Efficiency is not provided in {self.file_path}, set to 1.")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load _anti_correlation_eff.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/_anti_correlation_eff.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_

AmBe_llh's map s2_cut_acc is using the parameter s2_cut_acc_sigma.
AmBe_llh's map s1_cut_acc is using the parameter s1_cut_acc_sigma.
AmBe_llh's map s1_eff_n_hits_3f is using the parameter s1_eff_n_hits_3f_sigma.


In [10]:
#apt.clear_cache() # to clear cache, if you want to re-compile the component

In [11]:
# apt.share._cached_configs, apt.share._cached_functions

In [12]:
nr.show_config()

,option,default,current,applies_to,help
0,s1_bias_3f,_s1_bias.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s1_area],3fold S1 reconstruction bias
1,s1_smear_3f,_s1_smearing.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s1_area],3fold S1 reconstruction smearing
2,s1_correction,_s1_correction.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s1_correction],S1 xyz correction on reconstructed positions
3,s1_lce,_s1_correction.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s1_lce],S1 light collection efficiency
4,g4,"[g4_ambe_sr0.npy, 1000000, 7]","[g4_ambe_sr1.npy, 100000, 7]","[energy, x, y, z, time, s2_tag, event_id, g4_e...",full-chain geant4 input file
5,literature_field,23.0,23.0,[ThomasImel],Drift field in each literature
6,s2_bias,_s2_bias.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s2_area_naive],S2 reconstruction bias
7,s2_smear,_s2_smearing.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[s2_area_naive],S2 reconstruction smearing
8,s2_bias,_s2_bias.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[alt_s2_area_naive],S2 reconstruction bias
9,s2_smear,_s2_smearing.json,/home/puehlengt/private_nt_aux_files/ntauxfile...,[alt_s2_area_naive],S2 reconstruction smearing


In [13]:
# For reference, this is the compiled code, the function is stored in appletree.share._cached_functions
# Initialize component
print('NR')
print(nr.code)

NR
from functools import partial
from jax import jit
from appletree.plugins import BootstrapMS
from appletree.plugins import ThomasImelBox
from appletree.plugins import QyNR
from appletree.plugins import TotalQuanta
from appletree.plugins import LyNR
from appletree.plugins import MeanNphNe
from appletree.plugins import MeanExcitonIon
from appletree.plugins import TrueExcitonIonNR
from appletree.plugins import OmegaNR
from appletree.plugins import MSDriftLoss
from appletree.plugins import TruePhotonElectronNR
from appletree.plugins import MSElectronDrifted
from appletree.plugins import MSElectronMerge
from appletree.plugins import MSPositionRecon
from appletree.plugins import MSS2LCEAlt
from appletree.plugins import MSS2PEAlt
from appletree.plugins import MSS2CorrectionAlt
from appletree.plugins import MSS2Alt
from appletree.plugins import MSS2LCEMain
from appletree.plugins import MSS2PEMain
from appletree.plugins import MSS2CorrectionMain
from appletree.plugins import MSS2Main
from app

In [14]:
# nr.needed_parameters

## Simulation

In [15]:
# Of course we have to load parameters (and their priors) in simulation (who the hell writes such comments..)
par_manager = apt.Parameter(get_file_path(instruct['par_config']))

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load param_nr_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/param_nr_sr1.json
  warn(f"Load {fname} successfully from {fpath}")


In [16]:
batch_size = int(1e4) # for funsies. design flaw, batch_size doesn't do shits here for multiple scatters like AmBe

#num_sims = int(3000)
num_sims = int(100)

param_bag = []
events_bag = []

for _mc in range(num_sims):
    key = apt.randgen.get_key(seed=_mc)

    par_manager.sample_prior() # sampling from prior
    parameters = par_manager.get_all_parameter()

    # simulate
    key, (cs1, cs2, eff) = nr.simulate(key, batch_size, parameters)

    # randomly sampling number of events according to ambe_nr_rate parameter
    n = sps.poisson.rvs(mu=parameters['ambe_nr_rate'])

    # must normalise
    norm_eff = np.array(eff)
    norm_eff /= norm_eff.sum()

    # randomly sample n events from all sim events according to weights, norm_eff
    sel_ind = np.random.choice(len(eff), size=n, p=norm_eff, replace=False) # np reweighs weights for me
    sel_cs1, sel_cs2 = np.array(cs1[sel_ind]), np.array(cs2[sel_ind])
    events = np.vstack((sel_cs1, sel_cs2))

    # store things
    param_bag.append(parameters.copy())
    events_bag.append(events)
    

In [17]:
# todo:
# [done] 0. realistic parameters file
# 1. save the simulated data
# [done] 2. visualize the simulated data
# [done, i think?] 3. think of a way to sample number of events according to the rate parameter

In [20]:
per_sim = (53-40.)/(99-29)
target = 200_000

per_sim*target/60./60. # hours

10.317460317460318

In [18]:
raise

RuntimeError: No active exception to reraise

In [ ]:
max_plots = 10

cnt = 0
for _mc in range(num_sims):
    if cnt >= max_plots:
        break
    this_params = param_bag[_mc]
    this_events = events_bag[_mc]

    plt.figure()
    plt.hist2d(this_events[0,:], this_events[1,:],
            bins=[np.linspace(0.1, 150, 100), np.geomspace(10**2, 10**4, 100)], norm=LogNorm())
    plt.xlabel('cs1 [PE]')
    plt.ylabel('cs2 [PE]')
    plt.yscale('log')
    plt.title(_mc)

    cnt += 1

In [ ]:
batch_size = int(1e4) # design flaw, batch_size doesn't do shits here for multiple scatters like AmBe

key = apt.randgen.get_key()
key, (cs1, cs2, eff) = nr.simulate(key, batch_size, parameters)


# length of cs1, cs2, eff the same each time (but is not the same as batch_size), but the sum of eff changes (thankfully)

In [ ]:
num_sigmas = 6
tail_prob = sps.norm.sf(x=num_sigmas)
suggested_max_batch = sps.norm.ppf(1-tail_prob,
                                   loc=par_manager.par_config['ambe_nr_rate']['init_mean'],
                                   scale=par_manager.par_config['ambe_nr_rate']['init_mean'])
print(suggested_max_batch) # number of NR events hardly gonna fluctuate above this

In [ ]:
plt.hist2d(sel_cs1, sel_cs2, weights=eff[sel_ind], bins=[np.linspace(0.1, 150, 100), np.geomspace(10**2, 10**4, 100)], norm=LogNorm())
plt.yscale('log')

In [ ]:
n_events_selected = _cached_functions['AmBe_llh']['BootstrapMS_AmBe'].g4.n_events_selected # tbh what is this again?

In [ ]:
key, (cs1, cs2, eff) = nr.simulate(key, batch_size, parameters)

In [ ]:
target = 200_000

40/num_sims*target/60./60. # hours